In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# 1. Cargar dataset original
df = pd.read_csv('bank-full.csv', sep=';')

# Divide el dataset en 2 partes: 90% y 10%
df_90, df_10 = train_test_split(df, test_size=0.10, random_state=42)
df_10.to_csv('dataset_10_streaming.csv', index=False) # Guarda el 10% para el Productor

# 2. Prepara train, test y validación
X = df_90.drop('y', axis=1)
y = df_90['y'].apply(lambda x: 1 if x == 'yes' else 0)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Crear pipeline de preprocesamiento y modelo
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

modelo = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Entrenar
modelo.fit(X_train, y_train)

# 3. Guardar modelo
joblib.dump(modelo, 'modelo_bankfull.pkl')
print("✅ Parte 1 completada: Modelo entrenado y datos guardados.")

In [ ]:
import pandas as pd
import json
import joblib
from kafka import KafkaConsumer

modelo = joblib.load('modelo_bankfull.pkl')

consumer = KafkaConsumer(
    'bank_topic', 
    bootstrap_servers=['localhost:9092'], 
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

UMBRAL = 0.65 

print("🎧 Escuchando mensajes de Kafka... (Esta celda se quedará ejecutándose)")

for message in consumer:
    datos = message.value
    df_predict = pd.DataFrame([datos])
    
    prob_yes = modelo.predict_proba(df_predict)[0][1] 
    resultado = "CONTRATA" if prob_yes >= UMBRAL else "NO CONTRATA"
    
    print("-" * 40)
    print(f"👤 Cliente: {datos.get('age')} años, {datos.get('job')}, {datos.get('duration')}s de llamada")
    print(f"📊 Probabilidad: {prob_yes:.2f} -> Decisión: {resultado}")